<a href="https://colab.research.google.com/github/Laurel-Benedict/Col-lethal/blob/main/depmap_tcga.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

In [ ]:
path = "/content/drive/MyDrive/depmap-tcga/CRISPRGeneEffect.csv"

df = pd.read_csv(path, index_col=0, low_memory=False)

In [ ]:
print(df.shape)
print(df.columns[:5])
print(df.index[:5])

(1186, 18435)
Index(['A1BG (1)', 'A1CF (29974)', 'A2M (2)', 'A2ML1 (144568)',
       'A3GALT2 (127550)'],
      dtype='object')
Index(['ACH-000015', 'ACH-000045', 'ACH-000094', 'ACH-000096', 'ACH-000099'], dtype='object')


In [ ]:
df.columns = df.columns.str.replace(r" \(.*\)", "", regex=True)

In [ ]:
print(df.columns[:5])

Index(['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2'], dtype='object')


In [ ]:
dups = df.columns[df.columns.duplicated()]
print(dups)

Index([], dtype='object')


In [ ]:
df = df.loc[:, ~df.columns.duplicated()]

In [ ]:
print(df.shape)
print(df.columns[:5])
print(df.index[:5])

(1186, 18435)
Index(['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2'], dtype='object')
Index(['ACH-000015', 'ACH-000045', 'ACH-000094', 'ACH-000096', 'ACH-000099'], dtype='object')


In [ ]:
path1= "/content/drive/MyDrive/depmap-tcga/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"
df1 = pd.read_csv(path1, index_col=0, low_memory=False)

In [ ]:
print(df1.columns[:10])

Index(['SequencingID', 'ModelConditionID', 'ModelID', 'IsDefaultEntryForMC',
       'IsDefaultEntryForModel', 'TSPAN6 (7105)', 'TNMD (64102)',
       'DPM1 (8813)', 'SCYL3 (57147)', 'FIRRM (55732)'],
      dtype='object')


In [ ]:
df1 = df1.set_index("ModelID")

In [ ]:
df1 = df1.select_dtypes(include='number')

In [ ]:
df1.columns = df1.columns.str.replace(r" \(.*\)", "", regex=True)

In [ ]:
df1 = df1.select_dtypes(include='number')

In [ ]:
print(df1.shape)
print(df1.columns[:5])
print(df1.index[:5])

(1775, 19215)
Index(['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'FIRRM'], dtype='object')
Index(['ACH-001113', 'ACH-001289', 'ACH-001339', 'ACH-001619', 'ACH-001979'], dtype='object', name='ModelID')


In [ ]:
X_depmap = df1  # (cell lines × genes)
Y_depmap = df # (cell lines × genes)

In [ ]:
target_gene = "A1BG"
print(target_gene in Y_depmap.columns)

True


In [ ]:
valid_genes = Y_depmap.columns

print(len(valid_genes))
print(valid_genes[:20])

18435
Index(['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2', 'A4GALT', 'A4GNT', 'AAAS',
       'AACS', 'AADAC', 'AADACL2', 'AADACL3', 'AADACL4', 'AADAT', 'AAGAB',
       'AAK1', 'AAMDC', 'AAMP', 'AANAT', 'AAR2'],
      dtype='object')


In [ ]:
def clean_cell_names(index):
    return (
        index.str.strip()
             .str.replace("_", "-", regex=False)
             .str.split(".").str[0]
             .str.split("-").str[:2].str.join("-")  # keep ACH-xxxxx
    )

X_depmap.index = clean_cell_names(X_depmap.index.to_series())
Y_depmap.index = clean_cell_names(Y_depmap.index.to_series())

In [ ]:
common_cells = X_depmap.index.intersection(Y_depmap.index)

print("Common cells:", len(common_cells))
X_depmap = X_depmap.loc[common_cells]
Y_depmap = Y_depmap.loc[common_cells]

Common cells: 1118


In [ ]:
# Remove duplicate cell lines in X
X_depmap = X_depmap[~X_depmap.index.duplicated(keep='first')]

In [ ]:
# Find common cell lines (samples)
common_cells = X_depmap.index.intersection(Y_depmap.index)

# Filter both datasets
X_depmap = X_depmap.loc[common_cells]
Y_depmap = Y_depmap.loc[common_cells]

print("Aligned X:", X_depmap.shape)
print("Aligned Y:", Y_depmap.shape)

Aligned X: (1118, 19215)
Aligned Y: (1118, 18435)


In [ ]:
print(X_depmap.index.equals(Y_depmap.index))

True


In [ ]:
print(X_depmap.index[:10])
print(Y_depmap.index[:10])

Index(['ACH-001289', 'ACH-001619', 'ACH-001538', 'ACH-000242', 'ACH-000233',
       'ACH-000461', 'ACH-001794', 'ACH-002023', 'ACH-000528', 'ACH-001655'],
      dtype='object')
Index(['ACH-001289', 'ACH-001619', 'ACH-001538', 'ACH-000242', 'ACH-000233',
       'ACH-000461', 'ACH-001794', 'ACH-002023', 'ACH-000528', 'ACH-001655'],
      dtype='object')


In [ ]:
scaler = StandardScaler()
X_depmap_scaled = scaler.fit_transform(X_depmap)

In [ ]:
for gene in Y_depmap.columns[:50]:
    y = Y_depmap[gene]

    lasso = Lasso(alpha=0.01, max_iter=10000)
    lasso.fit(X_depmap_scaled, y)

In [ ]:
coeffs = pd.Series(lasso.coef_, index=X_depmap.columns)

# Top important genes
print(coeffs.sort_values(ascending=False).head(10))
print(coeffs.sort_values().head(10))

USP29       0.004414
NIPAL2      0.002937
KRBOX5      0.002826
HPSE        0.001953
RAD51B      0.001381
KRT33A      0.001193
C1QB        0.001190
LIMA1       0.001144
ARHGAP10    0.001097
LRRC1       0.000883
dtype: float64
COL4A4   -0.002504
TCAP     -0.001138
SPDYE8   -0.001061
MRPS33   -0.001048
RETNLB   -0.000991
GUCA1C   -0.000917
TMUB1    -0.000822
SLC3A1   -0.000785
SPMAP1   -0.000708
GABRA2   -0.000702
dtype: float64


In [ ]:
nonzero = (lasso.coef_ != 0).sum()
print("Number of selected genes:", nonzero)

Number of selected genes: 41


In [ ]:
from sklearn.metrics import r2_score

y_pred_train = lasso.predict(X_depmap_scaled)

r2 = r2_score(y, y_pred_train)
print("R² score:", r2)

R² score: 0.05639144222352199
